In [1]:
# 1. Libraries
import pandas as pd
from sklearn.model_selection import train_test_split

# 2. Load the synthetic dataset
df = pd.read_csv("../data/synthetic_pain_patients.csv")

# 3. Safety gate: patients with red flags are referred, not classified
df_model = df[~df["any_red_flag"]].copy()

print("All patients:", len(df))
print("Excluded (red flags):", df["any_red_flag"].sum())
print("Eligible for the model:", len(df_model))

# 4. Features (X) and target (y)
NUMERIC_FEATURES = [
    "pain_duration_months", "dn4_score", "paindetect_score", "csi_score",
    "sleep_quality", "hads_anxiety", "hads_depression", "pcs_score",
    "tsk_score", "work_stress",
]
CATEGORICAL_FEATURES = ["pain_distribution", "imaging_tissue_finding", "imaging_nerve_compression"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "mechanism"

X = df_model[FEATURES]
y = df_model[TARGET]

# 5. Train/test split (80/20, stratified by mechanism)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

All patients: 2000
Excluded (red flags): 190
Eligible for the model: 1810
Train: (1448, 13) | Test: (362, 13)


In [2]:
# Check: class proportions in train and test
pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).round(3)

,train,test
mechanism,,
nociceptive,0.394,0.395
mixed,0.264,0.262
nociplastic,0.190,0.191
neuropathic,0.152,0.152


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Build the preprocessing pipeline (with or without missing indicators)
def make_preprocessor(add_missing_indicator):
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=add_missing_indicator)),
        ("scale", StandardScaler()),
    ])
    categorical_steps = OneHotEncoder(handle_unknown="ignore")

    return ColumnTransformer([
        ("num", numeric_steps, NUMERIC_FEATURES),
        ("cat", categorical_steps, CATEGORICAL_FEATURES),
    ])


preprocessor_plain = make_preprocessor(add_missing_indicator=False)
preprocessor_indicator = make_preprocessor(add_missing_indicator=True)

# Learn imputation, scaling and encoding from the TRAINING data only
X_train_plain = preprocessor_plain.fit_transform(X_train)
X_train_indicator = preprocessor_indicator.fit_transform(X_train)

print("Without missing indicators:", X_train_plain.shape)
print("With missing indicators:   ", X_train_indicator.shape)

Without missing indicators: (1448, 20)
With missing indicators:    (1448, 27)


In [4]:
# Names of the columns created by the preprocessing
list(preprocessor_indicator.get_feature_names_out())

['num__pain_duration_months',
 'num__dn4_score',
 'num__paindetect_score',
 'num__csi_score',
 'num__sleep_quality',
 'num__hads_anxiety',
 'num__hads_depression',
 'num__pcs_score',
 'num__tsk_score',
 'num__work_stress',
 'num__missingindicator_dn4_score',
 'num__missingindicator_paindetect_score',
 'num__missingindicator_csi_score',
 'num__missingindicator_hads_anxiety',
 'num__missingindicator_hads_depression',
 'num__missingindicator_pcs_score',
 'num__missingindicator_tsk_score',
 'cat__pain_distribution_dermatomal',
 'cat__pain_distribution_localized',
 'cat__pain_distribution_regional',
 'cat__pain_distribution_widespread',
 'cat__imaging_tissue_finding_no',
 'cat__imaging_tissue_finding_not_performed',
 'cat__imaging_tissue_finding_yes',
 'cat__imaging_nerve_compression_no',
 'cat__imaging_nerve_compression_not_performed',
 'cat__imaging_nerve_compression_yes']